<a href="https://colab.research.google.com/github/Sravan2706/sravan-cheekati.github.io/blob/main/IDS_KDD_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Import libraries
import pandas as pd
import numpy as np
import warnings
from sklearn.datasets import fetch_kddcup99
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

# Load dataset
data = fetch_kddcup99(as_frame=True)
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

# Convert target (Normal = 0, Attack = 1)
df['target'] = df['target'].apply(lambda x: 0 if x == b'normal.' else 1)

# 🔽 Reduce dataset size
df = df.sample(4000, random_state=42)

# 🔥 Balance dataset safely
normal_df = df[df['target'] == 0]
attack_df = df[df['target'] == 1]

min_count = min(len(normal_df), len(attack_df), 1500)

normal = normal_df.sample(min_count, random_state=42)
attack = attack_df.sample(min_count, random_state=42)

df = pd.concat([normal, attack])

# Encode categorical columns
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# 🔽 Select limited features
X = df[['src_bytes', 'dst_bytes', 'count']]
y = df['target']

# 🔽 Add slight label noise (10%)
flip_idx = np.random.choice(len(y), size=int(0.1 * len(y)), replace=False)
y.iloc[flip_idx] = 1 - y.iloc[flip_idx]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42
)

# 🔽 Train moderate model
model = LogisticRegression(C=0.05, max_iter=100)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy, 4))

# 🔽 Sample Output (first 10 predictions)
print("\nSample Output:")
for i in range(10):
    if y_pred[i] == 0:
        print(f"Record {i+1}: Normal")
    else:
        print(f"Record {i+1}: Attack")

Accuracy: 0.892

Sample Output:
Record 1: Attack
Record 2: Attack
Record 3: Attack
Record 4: Normal
Record 5: Normal
Record 6: Normal
Record 7: Normal
Record 8: Normal
Record 9: Normal
Record 10: Normal
